# 課程 11 - 代理對代理 (A2A) 協議


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## 創建專門的旅行代理人


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## 透過工作流程進行多智能體協作

我們將三個智能體連接成一個反映 A2A 訊息傳遞的順序工作流程：

1. **CurrencyExchangeAgent** 接收用戶請求並提供貨幣指引。
2. **ActivityPlannerAgent** 接收豐富的上下文並加入活動建議。
3. **TravelManagerAgent** 將兩者輸入綜合成最終的旅行簡報。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 理解生產環境中的 A2A

在生產環境中，A2A 協議解鎖了強大的跨服務場景：

| 功能 | 描述 |
|---|---|
| <strong>跨框架互操作</strong> | 使用一個框架構建的代理可以將任務委派給任何其他符合 A2A 規範的框架所構建的代理，實現真正的跨組織互操作性。 |
| <strong>服務邊界</strong> | 代理可以存在於獨立的微服務、雲區域，甚至不同組織中，同時仍能無縫協作。 |
| <strong>動態發現</strong> | 協調器可以在運行時查詢代理卡註冊表，尋找最適合特定子任務的專家。 |
| <strong>串流與推送通知</strong> | A2A 支援伺服器發送事件 (SSE) 用於即時進度更新，以及長時間運行任務的推送通知。 |

我們之前構建的工作流程是此模式的簡化版、同進程版本。在真實
部署中，每個代理都會公開 HTTP 端點，發佈代理卡，並通過 A2A JSON-RPC 協議
進行通訊。


## 總結

在本課程中你學到了：

1. **什麼是 A2A 協議** — 一個用於代理人間發現、消息傳遞和任務管理的開放標準。
   和任務管理。
2. <strong>如何創建專門的代理人</strong> — 一個貨幣兌換代理、一個活動規劃代理，
   和一個旅遊管理協調器。
3. <strong>如何將代理人接入工作流程</strong> — 使用 `WorkflowBuilder` 來建模代理人間連續的消息傳遞。
   消息傳遞。
4. **A2A 在生產環境中的運作方式** — 支持跨框架、跨服務的協作，具備動態發現和流式更新功能。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
